In [ ]:
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
from matplotlib.pylab import rcParams
from statsmodels.tsa.stattools import adfuller
from pathlib import Path
import pmdarima as pm
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

windspeed_df = pd.read_csv(str(ROOT / "data/DMI_ws.csv"))
irr_df = pd.read_csv(str(ROOT / "data/DMI_radiation.csv"))
PV_df = pd.read_csv(str(ROOT / "data/SyslabPV_15min_collective_PV.csv"))
PV_df["ts"] = pd.to_datetime(PV_df['ts'])
PV_df = PV_df.set_index("ts")
PV_df = PV_df
Wind_df = pd.read_csv(str(ROOT / "data/SyslabWind_15min_nozeros.csv"))
df = pd.read_csv(str(ROOT / "data/combined_data_positive_gen.csv"))

In [ ]:
seasonality = 24*4 
PV_df["Power diff"] = PV_df['Collective PV'].diff(periods=seasonality)

In [ ]:
result = seasonal_decompose(PV_df['Collective PV'].dropna(), model='addiditve', period=seasonality)
trend = result.trend.dropna()
seasonal = result.seasonal.dropna()
residual = result.resid.dropna()

In [ ]:

# Plot the decomposed components
plt.figure(figsize=(6,6))
#x_labels = [t.strftime('%m %Y') for t in PV_df.index]
plt.subplot(4, 1, 1)
plt.plot(PV_df.index,PV_df['Collective PV'], label='Original Series')

#plt.xticks(range(0, PV_df.size, 2880), x_labels[::2880], rotation=45)
plt.legend()

plt.subplot(4, 1, 2)
plt.plot(trend, label='Trend')
plt.legend()

plt.subplot(4, 1, 3)
plt.plot(seasonal, label='Seasonal')
plt.legend()

plt.subplot(4, 1, 4)
plt.plot(residual, label='Residuals')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
windspeed_df = pd.read_csv(str(ROOT / "data/DMI_ws.csv"))
irr_df = pd.read_csv(str(ROOT / "data/DMI_radiation.csv"))
irr_df["ts"] = pd.to_datetime(irr_df['ts'])
irr_df = irr_df.set_index("ts")


PV_df = pd.read_csv(str(ROOT / "data/SyslabPV_60min.csv"))
PV_df["ts"] = pd.to_datetime(PV_df['ts'])
PV_df = PV_df.set_index("ts")


Wind_df = pd.read_csv(str(ROOT / "data/SyslabWind_15min_nozeros.csv"))
df = pd.read_csv(str(ROOT / "data/combined_data_positive_gen.csv"))

In [ ]:
def plot_predicted_vs_measured(measured,predicted,dataset):
    # Scatter plot: measured vs predicted on test set
    plt.figure(figsize=(6, 6))
    plt.scatter(measured, predicted, alpha=0.6)
    plt.xlabel("Measured power")
    plt.ylabel("Predicted power")
    plt.title(f"{dataset} model: measured vs predicted (test set)")

    min_val = min(measured.min(), predicted.min())
    max_val = max(measured.max(), predicted.max())
    plt.plot([min_val, max_val], [min_val, max_val])

    r2 = r2_score(measured, predicted)
    mae = mean_absolute_error(measured, predicted)
    rmse = mean_squared_error(measured, predicted) ** 0.5

    print("Wind model")
    print("R²:", r2)
    print("MAE:", mae)
    print("RMSE:", rmse)

    plt.show()

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

def Sarimax(Type,file_path,endo_name,exo_name,granularity=10,order = (1,0,1),seasonal_order = (1,1,1),gap = ["2025-06-14 00:00:00","2025-06-21 00:00:00"]):
    """Using the SARIMAX function to fill gaps in the endogenous variable using the exogeneous variable
    Granularity is the amount of time between measurements in minutes
    Type is either "Wind" or "Solar" """


    try:
        df = pd.read_csv(file_path,delimiter=",")
    except Exception as e:
        print(f"Error reading file: {e}")
        return
    df["ts"] = pd.to_datetime(df["ts"])
    df = df.set_index("ts") # Set datetime as index
    df = df.loc[df.index >"2025-05-14 07:00:00"] # start from when our power data starts

    df_saved = df.copy()

    df.loc[df[exo_name].isna(),exo_name] = 0 # REMOVE ANY NANS FROM EXO VARIABLE AND REPLACE WITH 0.... 

    if Type == "Wind":
        # Wind power is proportional to the cubic wind speed
        gap_measured = df.loc[(gap[0] < df.index) & (df.index < gap[1]),endo_name] # save the data that we will later use for testing
        df.loc[(gap[0] < df.index )& (df.index < gap[1]),endo_name] = np.nan
        df["active"] = (df[exo_name]> 3).astype(int)
        df["w3"] = df[exo_name]**3
        exo = df[[exo_name,"active","w3"]]
        print("Preparing Model for Wind Turbine")
    elif Type == "Solar":
        df = df.loc[df[exo_name] >= 1] # solar power remove night time!
        gap_measured = df.loc[(gap[0] < df.index) & (df.index < gap[1]),endo_name] # save the data that we will later use for testing
        df.loc[(gap[0] < df.index )& (df.index < gap[1]),endo_name] = np.nan
        exo = df[exo_name] 
        print("Preparing Model for PV unit")


    
    endo = df[endo_name]

    seasonality = 24*60/granularity
    seasonal_order += seasonality , 
    model = SARIMAX(
        endo,
        exog=exo,
        order=order,
        seasonal_order=seasonal_order,  # daily seasonality
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    results = model.fit()

    endo_modelled = results.fittedvalues
    endo_filled = endo.copy()
    endo_filled.loc[endo.isna()] = endo_modelled.loc[endo.isna()]
    
    # results.plot_diagnostics()



    #if Type == "Solar":
    #    df_saved[df_saved[exo_name] >= 1] = endo_filled # redo the zero rows that were removed because irradiance was zero

    endo_filled.to_csv(f'{file_path[:-4]}_{endo_name}_SARIMAX_{order}{seasonal_order}.csv', index=True)

    gap_predicted = endo_filled[(gap[0] < df.index) & (df.index < gap[1])]

    plot_predicted_vs_measured(gap_measured,gap_predicted,endo_name)



In [ ]:
combined_data = str(ROOT / "data/combined_data_DMI_gen.csv")
combined_df = pd.read_csv(combined_data,delimiter=",")
combined_df["ts"] = pd.to_datetime(combined_df["ts"])
combined_df = combined_df.set_index("ts")


In [ ]:
combined_df["Global Irradiance"]

In [ ]:
df_saved = combined_df.copy()
combined_df = combined_df.loc[combined_df["Global Irradiance"] >= 1]
df_saved[df_saved["Global Irradiance"]>= 1] = combined_df #### REPLACE WITH RESULTS 

In [ ]:
df_saved

In [ ]:
Sarimax("Wind",combined_data,"Aircon_WT Power","Windspeed",granularity = 60,order = (1,1,5),seasonal_order = (1,1,1))

In [ ]:
Sarimax("Solar",combined_data,"B715","Global Irradiance",granularity = 60,order = (1,1,1),seasonal_order = (1,1,1))